# Reproduce the portfolio validity results

This notebook checks the included data, rebuilds the result tables and verifies the main reported quantities.


In [ ]:
from pathlib import Path
import os
import subprocess

repo_url = 'https://github.com/evidenceworks/university-portfolio-validity'
repo_name = 'university-portfolio-validity'
cwd = Path.cwd()

if (cwd / 'code' / 'check.py').is_file():
    repo = cwd
elif (cwd / repo_name / 'code' / 'check.py').is_file():
    repo = cwd / repo_name
else:
    base = Path('/content') if Path('/content').is_dir() else cwd
    repo = base / repo_name
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo)], check=True)

os.chdir(repo)
print(f'Repository: {repo}')


## Check and reproduce

The data are checked before and after deterministic regeneration of the two result tables.


In [ ]:
for command in (
    ['python', 'code/check.py'],
    ['python', 'code/reproduce.py'],
    ['python', 'code/check.py'],
):
    print('$', ' '.join(command))
    subprocess.run(command, check=True)


## Results

The final cell displays the principal portfolio validity results and the descriptive robustness and sensitivity diagnostics.


In [ ]:
import csv

with open('results/summary.csv', encoding='utf-8', newline='') as handle:
    summary = {row['metric']: row['value'] for row in csv.DictReader(handle)}

display_metrics = (
    ('Audited cases', 'audited_cases'),
    ('Grade A', 'valid_a'),
    ('Grade B', 'valid_b'),
    ('Invalid', 'invalid'),
    ('Accepted', 'accepted'),
    ('Eligible', 'eligible'),
    ('Membership difference rate', 'membership_difference_rate'),
    ('Jaccard overlap', 'jaccard_overlap'),
    ('Accepted precision', 'accepted_precision'),
    ('Eligible retention', 'eligible_retention'),
    ('Exact reviewer agreement rate', 'reviewer_exact_agreement_rate'),
    ('Collapsed reviewer agreement rate', 'reviewer_collapsed_agreement_rate'),
    ('Source-labeled conference papers', 'conference_source_labeled'),
    ('Conference papers excluded by the historical type rule', 'conference_excluded_by_type'),
    ('Adjudicated eligible conference-paper cases', 'conference_eligible'),
)

for label, metric in display_metrics:
    print(f'{label}: {summary[metric]}')

print('\nRobustness and sensitivity diagnostics')
with open('results/robustness.csv', encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    values = [row['analysis'], row['group']]
    for field in ('n', 'accepted_precision', 'eligible_retention', 'lower', 'upper', 'replicates'):
        if row[field] != '':
            values.append(f'{field}={row[field]}')
    print(' | '.join(values))
